# 02 · Clinical sets, Shapley blocks and checks
**CVES manuscript repair series** · supervisor's repair guide (2 Nov 2026)

| | |
|---|---|
| **Guide steps** | 7 (explainability), 8 (clinical table) |
| **Needs** | Notebook 00 done |
| **Produces** | `3_clinical`, Table 3 (n = 64 and n = 91), Shapley table, coefficient figure |
| **Runtime** | about 3–6 minutes |
| **Done when** | the 17-variable numbers are gone, both cohorts are shown, and one interval is used (refit out-of-bag) |

**How to use.** Run the cells top to bottom. Each notebook has the same five parts: **0 Setup** → **1 Run the kit step** → **2 Compare with the supervisor's reference run** → **3 Tables and figures** → **4 Where each number goes**.
All numbers are *exploratory* until regenerated from a frozen commit (notebook 05). Everything printed here is an aggregate; per-participant files stay in the private folder.

In [ ]:
#@title 0. Setup (run in every session) { display-mode: "form" }
from google.colab import drive
drive.mount('/content/drive')
!pip -q install wfdb
import sys; sys.path.insert(0, '/content/drive/MyDrive/CVES Fresh/CVES Repair/kit')
import helpers as H
cfg = H.setup()

## 1 · Run the kit step
Clinical variable sets: **R** (10 pre-stroke risk factors), **P** (5 post-stroke proxies: gait speed and four medication classes), **A** = R + P. Regularised logistic regression with `C = 1.0`, subject-grouped cross-validation, refit out-of-bag bootstrap, leave-one-out, TIA exclusion, fold-wise vs all-data preprocessing, shuffled-label null, calibration slope, and the all-completed-visits cohort (n = 91). Hypertension years are recoded (NO → 0) inside `clinical.build`.

In [ ]:
#@title 1. Clinical sets, intervals, Shapley, checks { display-mode: "form" }
H.run_step(cfg, '3_clinical', 'step3_clinical.py', cfg.data)

In [ ]:
#@title 2. Compare with the supervisor's reference run { display-mode: "form" }
H.compare(cfg, '3_clinical')
# Rule of thumb from the guide: counts exact, AUC +/-0.01, phase and timing +/-1 degree.
# Differences beyond that are listed above. Library versions can cause small ones; report the rest as implementation sensitivity.

## 3 · Tables and figures

In [ ]:
#@title 3a. Table 3: AUC by variable set, n = 64 and n = 91 { display-mode: "form" }
t3 = H.read_report(cfg, '3_clinical')
t7 = H.read_report(cfg, '7_robustness') if (cfg.reports / '7_robustness.txt').exists() else ''
ct = H.clinical_tables(t3, t7)
ct['auc'].round(3)

In [ ]:
#@title 3b. Shapley blocks on out-of-fold AUC above 0.5 { display-mode: "form" }
if len(ct['shapley']):
    display(ct['shapley'])
    print('Both cohorts come from 7_robustness (notebook 04). The order of gait and medications changes between cohorts: do not call gait the clear leader.')
else:
    print('n = 91 Shapley blocks appear after notebook 04 runs 7_robustness. For now, the n = 64 version:')
    print(H.grep(t3, r'^Shapley'))

In [ ]:
#@title 3c. Standardised coefficients with bootstrapped rank and sign { display-mode: "form" }
display(ct['coef'])
H.coef_figure(ct['coef'], cfg.figs / 'fig_clinical_coefficients.png')
print(H.grep(t3, r'^age: marginal'))

In [ ]:
#@title 3d. Checks that must be reported { display-mode: "form" }
for pat, title in [(r'^n 64', 'Cohort and missing values'), (r'^refit OOB CI|^diffs', 'Intervals and paired differences'),
                   (r'^leave-one|^TIA', 'Leave-one-out and TIA exclusion'), (r'^fold-wise|^pooled', 'Preprocessing (leakage check)'),
                   (r'^shuffled|^calibration', 'Shuffled-label null and calibration'), (r'^strict-R|^retained', 'Strict-R sensitivity and retention')]:
    print('--', title); print(H.grep(t3, pat)); print()
print('Reminder: fold-wise and all-data preprocessing give 0.848 and 0.847, so this check cannot detect leakage here. Say that in the paper.')

## 4 · Where each number goes
| Manuscript | Take from |
|---|---|
| **Table 3** | `3a`: AUC for R, P, A at n = 64 and n = 91 (add the n = 91 column); the interval column is **refit out-of-bag** only. Delete the §4.3 sentence preferring the other interval |
| **R7, S4, §4.8** | `3b` (Shapley blocks, both cohorts), `3c` (coefficients: rank, sign), age marginal vs adjusted (−0.36 vs −0.84; stroke group is younger) |
| **§3.1, §4.1** | retention by group (86% stroke, 56% controls) and denominators per group |
| **Step 8** | recoded hypertension years; TIA exclusion; leave-one-out; the preprocessing line |
| **Remove** | every number from the old 17-variable fit and the in-sample permutation importance |

## 1 · Run the kit steps
Settings (from `ANALYSIS_LOCK.md`): last 300 s of the first seated run (sit-stand-balance, 1000 Hz) and 60–360 s for head-up tilt (**label this the supine baseline**); beats detected up to 5 s beyond the segment edge on an exact 300 s grid (5 windows); beat-to-beat means, 10 Hz linear interpolation, Welch 100 s Hann 50%, triangular smoothing, no detrend or filter; CARNet Table 2 gate (main), 200 surrogates and all-bins (sensitivity); per-record and regime-offset alignment; age + sex + EtCO₂ with HC3 and a t reference; within-regime permutation.

In [ ]:
#@title 1a. Signals: per-record phase and gain on all segments { display-mode: "form" }
# Writes private/step4.pkl (per-record; private) and reports/4_log.txt.
H.run_step(cfg, '4_log', 'step4_signals.py', cfg.data, cfg.work / 'step4.pkl')

In [ ]:
#@title 1b. The phase grid: clock x segment x gate x alignment x side { display-mode: "form" }
H.run_step(cfg, '4_grid', 'step4b_analysis.py', cfg.data, cfg.work / 'step4.pkl')

In [ ]:
#@title 1c. The draft's own contrast on the wrong vs the right clock (core exhibit) { display-mode: "form" }
H.run_step(cfg, '6_draft_clock', 'step4d_draft_clock.py', cfg.data)

In [ ]:
#@title 1d. Full curves for Figure S1 (group medians, 0.02-0.5 Hz) { display-mode: "form" }
H.run_step(cfg, '8_curves', 'curves.py', cfg.data, cfg.work / 'figS1_curves.csv')

In [ ]:
#@title 2. Compare with the supervisor's reference run (cell by cell) { display-mode: "form" }
import pandas as pd
mine = H.parse_grid(H.read_report(cfg, '4_grid'))
ref  = H.parse_grid((cfg.ref / '4_grid.txt').read_text())
k = ['proto', 'seg', 'side', 'gate', 'arm']
m = mine.merge(ref, on=k, suffixes=('', '_ref'))
m['d_est'] = m.b - m.b_ref
m['d_n'] = m.n - m.n_ref
print('Cells compared: %d' % len(m))
print('Same n: %d | n within 1: %d | largest n difference: %d' % ((m.d_n == 0).sum(), (m.d_n.abs() <= 1).sum(), m.d_n.abs().max()))
print('Estimate difference: median %.2f deg, within 1 deg: %d, within 3 deg: %d, largest %.1f deg' %
      (m.d_est.abs().median(), (m.d_est.abs() <= 1).sum(), (m.d_est.abs() <= 3).sum(), m.d_est.abs().max()))
ex = ((m.lo > 0) | (m.hi < 0)); ex_r = ((m.lo_ref > 0) | (m.hi_ref < 0))
print('Intervals excluding 0: yours %d, reference %d, cells where this differs: %d' % (ex.sum(), ex_r.sum(), (ex != ex_r).sum()))
H.compare(cfg, '6_draft_clock')

## 3 · Tables and figures

In [ ]:
#@title 3a. The core exhibit: same samples, wrong clock vs right clock { display-mode: "form" }
t6 = H.read_report(cfg, '6_draft_clock')
ce = H.core_exhibit(t6)
display(ce)
print(H.grep(t6, r'^windows per record|^Paired change|^  main gate|^  all bins|^correct clock'))
print()
print('How to read it (guide, Part 1): the draft\'s -11.2 degrees and low phase describe about 0.14-0.40 Hz, not low frequency.')
print('Report the band error as ESTABLISHED. Report the disappearance of the deficit as NOT DETECTED, with poor precision, not as disproved.')

In [ ]:
#@title 3b. Grid table (Table 5): main rows first300 and t60-360, sensitivity longest300 and HEADERCLOCK { display-mode: "form" }
g = H.parse_grid(H.read_report(cfg, '4_grid'))
print(len(g), 'cells parsed')
main = g[(g.seg.isin(['first300', 't60-360'])) & (g.gate == 'CARNet')].copy()
main['estimate'] = main.apply(lambda r: '%+.1f (%+.1f to %+.1f)' % (r.b, r.lo, r.hi), axis=1)
display(main.pivot_table(index=['protocol', 'seg', 'side'], columns='arm', values='estimate', aggfunc='first'))
print(H.grep(H.read_report(cfg, '4_grid'), r'^corrected-clock aligned cells|^header-clock cells'))

In [ ]:
#@title 3c. Claim rule (analysis lock RQ3): is a difference claimable anywhere? { display-mode: "form" }
rule = H.rq3_rule(g)
display(rule)
print('Claim a difference only if BOTH alignments exclude zero with the same sign under BOTH gates, in the seated session, and flag it as unconfirmed.')
print('Cells claimable:', int(rule.difference_claimable.sum()), 'of', len(rule), '. Otherwise write: "not detected, precision +/- X".')

In [ ]:
#@title 3d. Figure 4 (replacement): every grid cell { display-mode: "form" }
H.grid_forest(g, cfg.figs / 'fig4_phase_grid.png')
print('Saved', cfg.figs / 'fig4_phase_grid.png')

In [ ]:
#@title 3e. Figure S1: full gain, phase and coherence curves (CARNet recommends reporting these) { display-mode: "form" }
H.curves_figure(cfg.work / 'figS1_curves.csv', cfg.figs / 'figS1_curves.png')
print('Saved', cfg.figs / 'figS1_curves.png')

## 4 · Where each number goes
| Manuscript | Take from |
|---|---|
| **Lead exhibit, R1, A1** | `3a` (and the `HEADERCLOCK` rows of `3b` to show how the labelled-clock result depends on the segment: −3.5 on the first seated run, not −11.9) |
| **Table 5, Figure 4, §4.5** | `3b`, `3d`: report `first300` and `t60-360` as main, `longest300` and `HEADERCLOCK` as sensitivity; arms `un`, `pr`, `ro` |
| **R3** | `3c` and the precision lines (widest interval, robust estimates) in notebook 04 |
| **Figure S1, Table S1** | `figS1_curves.csv`, `3e` |
| **Wording** | "post-hoc", "aligned" (not "corrected"), "recording regime" (not "calibration setting"). Do not write "the clock error alone produces the deficit" |
| **Quote, do not regenerate** | the independent reimplementation differed by up to 8° at the main gate (guide, Honest limits). Say so in the paper |

## 1 · Run the kit steps

In [ ]:
#@title 1a. Increment test, reliability, three-link shortcut test { display-mode: "form" }
H.run_step(cfg, '5_tests', 'step4c_tests.py', cfg.data, cfg.work / 'step4.pkl')

In [ ]:
#@title 1b. Second-round checks (regime, order, bias, shortcut, repeatability, lab-table scale, mains line, Shapley) { display-mode: "form" }
H.run_step(cfg, '7_robustness', 'step5_robustness.py', cfg.data, cfg.work / 'step1.pkl', cfg.work / 'step4.pkl')

In [ ]:
#@title 2. Compare with the supervisor's reference run { display-mode: "form" }
H.compare(cfg, '5_tests')
H.compare(cfg, '7_robustness')
# Rule of thumb from the guide: counts exact, AUC +/-0.01, phase and timing +/-1 degree.
# Differences beyond that are listed above. Library versions can cause small ones; report the rest as implementation sensitivity.

In [ ]:
import pandas as pd
pd.set_option('display.max_rows', None, 'display.max_colwidth', None, 'display.width', 250)

for name in ['7_robustness', '5_tests']:
    df = H.compare(cfg, name)          # prints the match count and returns the table
    if df is not None and len(df):
        print('\n===== ' + name + ': all ' + str(len(df)) + ' flagged rows =====')
        for i, r in df.iterrows():
            print('\n[%d] %s' % (i, r['status']))
            print('  reference: ' + r['reference'])
            print('  yours:     ' + (r['yours'] or '(no matching line)'))

> `5_tests` also prints a **withdrawn** CO₂ comparison. The guide says not to report it: its windows mix CO₂ with time and protocol stage, so it cannot serve as a positive control.

## 3 · Tables, one block per manuscript item

In [ ]:
#@title 3a. Step 2 · One regime definition, the scale, and enrolment order (S2, R2) { display-mode: "form" }
t7 = H.read_report(cfg, '7_robustness'); t5 = H.read_report(cfg, '5_tests')
print(H.section(t7, 'one regime definition'))
print()
print(H.section(t7, 'instrumental rule'))
print()
print(H.section(t7, 'lab table vs stored'))
print()
print(H.section(t7, 'tilt sensitivity'))
print('\nSay: "regime and group are both confounded with enrolment sequence"; participant number as a proxy for time is an assumption.')

In [ ]:
#@title 3b. Step 3 · Measured regime effect vs pure delay, implied bias (R3) { display-mode: "form" }
for k in ['measured regime effect', 'effect of adding recording order', 'precision and robust']:
    print(H.section(t7, k)); print()
print('Give the bias two ways: pure delay, and measured with its interval (intervals reach about -14 degrees).')

In [ ]:
#@title 3c. Step 4 · Repeatability, without overclaiming (R4) { display-mode: "form" }
print(H.section(t7, 'reliability'))
print('\nLimit the claim to person-level use. Noise widens group intervals; it does not by itself bias them.')
print('The two seated runs differ in eyes open or closed, and supine vs seated differ in posture, so these are not pure repeat tests.')

In [ ]:
#@title 3d. Step 5 · Signal increment: no reliable gain (R5) { display-mode: "form" }
inc = H.increment_rule(t5)
display(inc)
print('Counts as an increment (refit CI above 0 AND gain above 0.04):', int(inc.counts_as_increment.sum()), 'of', len(inc))
print('Wording: "no reliable increment" (upper limits exceed 0.04), not "adds nothing". Test covers only phase and normalised gain.')

In [ ]:
#@title 3e. Step 6 · Non-circular shortcut test: regime defined from timing (R6) { display-mode: "form" }
print(H.section(t7, 'non-circular shortcut'))
print()
print('Check the negative control: ABP amplitude alone should NOT predict the timing regime (about 0.5).')
print('Claim to write: amplitude features recovered a recording change and separated groups little better than the regime alone;')
print('within-regime estimates are too imprecise to exclude disease signal. Do NOT write "not the disease".')

In [ ]:
#@title 3f. Step 7 · Shapley blocks, both cohorts (R7) { display-mode: "form" }
if not (cfg.reports / '3_clinical.txt').exists():
    print('Run notebook 02 first (it writes 3_clinical); Shapley blocks come from both reports.')
else:
    ct = H.clinical_tables(H.read_report(cfg, '3_clinical'), t7)
    display(ct['shapley'])

## 4 · Where each number goes
| Manuscript | Take from |
|---|---|
| **S2, R2, Table 2, Figure 3** | `3a` (one definition: left span on valid channels; scale 0.40 vs 1.02, ratio 2.53; OR seated 5.9, supine 9.5; order → regime 0.95, regime OR with order 0.26–0.45; ECG-to-ABP shift moves only in controls, so the offset is not called instrumental) |
| **R3** | `3b` |
| **R4** | `3c` (seated runs ρ about 0.17; same-person lag ρ 0.86) |
| **R5, Table 4** | `3d` (replace Table 4; Figure 5 goes) |
| **R6** | `3e` |
| **R7, S4** | `3f` |
| **Limitations** | enrolment-order confounding; no positive control; few controls in the high regime; implementation sensitivity; lab-table scale now filled in |

## 1 · Same-laboratory datasets: clocks only (guide, Step 12)
Run `external_headers.py` on *Cerebral Vasoregulation in Diabetes* and *Cerebral Perfusion and Cognitive Decline in Type 2 Diabetes*. It reads **headers and clocks only**, no group labels. Report one table: directory, header rate, heart rate, breathing.
The datasets are large. Put each in Drive (or download it to `/content`) and give the folder path below. Leave a path blank to skip it.

In [ ]:
#@title 1a. Clocks of the same-lab datasets { display-mode: "form" }
DIABETES_DIR = "" #@param {type:"string"}
PERFUSION_DIR = "" #@param {type:"string"}
for nm, dr in (('external_vasoregulation_diabetes', DIABETES_DIR), ('external_perfusion_t2d', PERFUSION_DIR)):
    if dr:
        H.run_step(cfg, nm, 'external_headers.py', dr)
    else:
        print('Skipped', nm, '(no folder given)')
print('\nSupervisor reference (reference/external_headers.txt):')
print((cfg.ref / 'external_headers.txt').read_text())

In [ ]:
#@title Ledger of every number in the draft { display-mode: "form" }
DRAFT_DOCX = "/content/drive/MyDrive/CVES Fresh/CVES_Stroke_Manuscript_BranchB.docx" #@param {type:"string"}
# Change the path to the draft you want listed. If the file is not found, an upload box appears.

import subprocess, shutil
from pathlib import Path
import pandas as pd

src = Path(DRAFT_DOCX)
if not src.exists():
    from google.colab import files
    print('Draft not found at that path. Upload the .docx now.')
    up = files.upload()
    src = Path('/content') / next(iter(up))

subprocess.run(['apt-get', '-qq', 'install', '-y', 'pandoc'], stdout=subprocess.DEVNULL, check=False)
subprocess.run(['pandoc', str(src), '-t', 'plain', '--wrap=none', '-o', str(cfg.work / 'draft.txt')], check=True)

H.run_step(cfg, 'ledger_summary', 'ledger.py', cfg.work / 'draft.txt', cfg.reports / 'ledger.csv')

L = pd.read_csv(cfg.reports / 'ledger.csv')
display(L.groupby('section', sort=False).size().rename('numbers').to_frame())

# Numbers from the superseded analysis: these must change or go
txt = (cfg.work / 'draft.txt').read_text(encoding='utf-8')
print('\nSuperseded items still in the draft (count):')
for k in ['11.2', '270 ms', '+0.08', '0.08', '10.4', 'leakage-aware', 'instrument setting', 'low-frequency']:
    print('  %-20s %d' % (k, txt.count(k)))
print('\nLedger saved to', cfg.reports / 'ledger.csv', '- open it in Sheets and fill the status and kit_report columns.')

## 3 · Number ledger (guide, Step 15)
The ledger lists every number in your draft with its section, so nothing is missed. Upload the draft as `.docx`, convert it with pandoc, and run `ledger.py`. Then trace the Abstract, Conclusion and main tables by hand, and batch the rest by kit report.

In [ ]:
#@title 3a. Ledger of every number in the draft { display-mode: "form" }
DRAFT_DOCX = "" #@param {type:"string"}
# Path to your draft .docx (in Drive or uploaded to /content). Leave blank to skip.
import subprocess
if DRAFT_DOCX:
    !apt-get -qq install -y pandoc > /dev/null
    subprocess.run(['pandoc', DRAFT_DOCX, '-t', 'plain', '--wrap=none', '-o', str(cfg.work / 'draft.txt')], check=True)
    H.run_step(cfg, 'ledger_summary', 'ledger.py', cfg.work / 'draft.txt', cfg.reports / 'ledger.csv')
    import pandas as pd
    L = pd.read_csv(cfg.reports / 'ledger.csv')
    display(L.groupby('section').size().rename('numbers').to_frame())
else:
    print('Skipped (no draft given).')

In [ ]:
#@title 3b. Status of each number: fill in `status` and `kit_report` in the CSV { display-mode: "form" }
if DRAFT_DOCX:
    print('Open', cfg.reports / 'ledger.csv', 'in Sheets. For each row set status (ok / changed / cut) and name the kit report it came from.')
    print('Your draft has about 900 numbers; the cuts remove many. Trace the Abstract, Conclusion and main tables first.')

## 4 · Freeze: manifest and package (guide, Step 16)

In [ ]:
#@title 4a. Manifest: software versions and a SHA-256 of every kit file, report and figure { display-mode: "form" }
import json
m = H.manifest(cfg)
print(json.dumps({k: m[k] for k in ('date', 'python', 'packages')}, indent=2))
print(len(m['kit']), 'kit files,', len(m['reports']), 'reports,', len(m['figures']), 'figures hashed.')
print('Saved', cfg.reports / 'MANIFEST.json')

In [ ]:
#@title 4b. Package the aggregates (kit + reports + figures; no per-participant files) { display-mode: "form" }
z = H.package(cfg, cfg.base / 'CVES_repair_aggregates.zip')
print('Wrote', z)
print('Excluded: private/*.pkl and the dataset.')

## Freeze and rerun checklist
1. **Commit** the kit, notebooks, `reports/` and `figures/` (not `private/`) to your repository, and write the commit hash into the Code availability statement (**S8**).
2. **Human rerun.** A *named author* opens a fresh Colab runtime, runs notebooks 00–05 from the frozen commit, reads the code, and checks every reported number against the ledger. Tolerance from the analysis lock: counts exact, AUC ±0.01, phase ±1°. After one reconciliation, report any remaining gap as *implementation sensitivity*.
3. **Quote, do not regenerate:** the independent reimplementation (up to 8° difference at the main gate; e.g. seated left unaligned −9.5° vs −1.4°). Say it came from a separate analysis that the kit cannot regenerate.
4. **Integrity text** S5–S9 (pre-specification, ethics, AI use, code, related work), author order and CRediT still need all authors.
5. **Still outside this notebook series:** trace the 38-record problem from your own notebook (print per-record bins passed, threshold and coherence, no identifiers); one statistician and one clinician read the paper before submission.